# Next-day ICU Transfer Prediction from Chest X-ray and Lab Data

This notebook trains and evaluates models that predict whether a patient is transferred from a non-ICU unit (ward/ER) to the ICU on the day after an index chest X-ray (CXR).

- **Cohort**: `icu_cohort.csv` from `02_build_icu_cohort.R` (MIMIC-IV OMOP CDM + MIMIC-CXR), one index CXR per patient
- **Label** (`outcome`): 0 = no transfer, 1 = next-day ICU transfer
- **Image branch**: frozen RAD-DINO encoder (`microsoft/rad-dino`) → one 768-dimensional embedding per CXR, extracted once and cached
- **Tabular branch**: 18 lab features + age + sex
- **Models**: a small fusion head on image embedding + tabular features, compared with a tabular-only head of the same design

**Pipeline**: configuration → cohort preprocessing (complete-case) → sample table → data check → view filter and 1:1 class-balanced sampling → visual check → stratified 70/15/15 split → RAD-DINO embedding extraction → tabular features → training with early stopping → test-set evaluation → error analysis

**Requirements**: authenticated access to the MIMIC-CXR GCS bucket, access to `microsoft/rad-dino` on Hugging Face, and a CUDA GPU (recommended; embedding extraction runs in fp16 on GPU and falls back to fp32 on CPU).

Reference: https://wikidocs.net/book/10449

---
### Step 0. Install packages

Run once in a terminal or in the cell below.

```bash
pip install -q numpy pandas opencv-python-headless torch transformers pillow pydicom gcsfs scikit-learn tqdm matplotlib
```

| Package | Purpose |
|---|---|
| numpy / pandas | Arrays and tables |
| opencv-python-headless | Image resizing for visual checks |
| torch | Embedding extraction, fusion model training |
| transformers | RAD-DINO image processor and encoder |
| pillow | Grayscale → RGB conversion for the RAD-DINO processor |
| pydicom | Reading DICOM headers and pixels |
| gcsfs | Reading DICOM files directly from Google Cloud Storage |
| scikit-learn | Data split, feature scaling, evaluation metrics |
| tqdm | Progress bars |
| matplotlib | Sample images and ROC curves |

In [ ]:
#!pip install -q numpy pandas opencv-python-headless torch transformers pillow pydicom gcsfs scikit-learn tqdm matplotlib

### Step 1. Imports and global configuration

All imports and every tunable constant live in this cell, so later cells contain no hard-coded paths or hyperparameters.

- **Files**: input cohort, preprocessed cohort, split tables, cached embeddings, model checkpoints and the error-analysis table
- **Image source**: DICOM files are read from GCS. `to_gcs_path()` keeps the `files/...` part of the cohort `local_path` and prefixes it with `GCS_BASE`.
- **Labels**: `TARGET_COL = "outcome"`, with `CLASS_NAMES[i]` naming label `i` (0 = no transfer, 1 = next-day ICU transfer)
- **Tabular features**: the 18 lab tests from `02_build_icu_cohort.R`. `REQ_COLS` are the columns that must be present for a patient to be kept (complete-case).
- **Sampling**: `N_POS` / `N_NEG` patients per class after the frontal-view filter (`ALLOWED_VIEWS = {PA, AP}`). `VALIDATE_ON_GCS = True` adds a file-integrity check (slower).
- **Image encoder**: `RAD_DINO` model id and its embedding size. `IMG_SIZE` is used only for the visual check; the model input size is set by the RAD-DINO processor.
- **Training**: split ratio, batch size, maximum epochs, early-stopping patience, optimizer and head settings

`torchaudio` is registered as unavailable before `transformers` is imported. Audio is not used, and this prevents a missing or broken `torchaudio` installation from breaking the import.

`set_seed()` fixes the Python, NumPy and PyTorch random seeds (and makes cuDNN deterministic on GPU), so sampling, splitting and training are reproducible.

In [ ]:
import os
import sys
import random
from typing import Optional
from functools import lru_cache
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import cv2
import pydicom
import gcsfs
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, accuracy_score, f1_score,
    classification_report, confusion_matrix, roc_curve,
)
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# Treat torchaudio as not installed (audio unused); must precede the transformers import
sys.modules["torchaudio"] = None
from transformers import AutoImageProcessor, AutoModel

# ── Files ──
COHORT_CSV       = "icu_cohort.csv"                  # output of 02_build_icu_cohort.R
PREPROCESSED_CSV = "icu_cohort_preprocessed.csv"     # complete-case cohort (Step 2)
SPLIT_CSV        = "icu_cohort_preprocessed_{}.csv"  # train / val / test tables (Step 10)
EMB_PREFIX       = "raddino"                         # embeddings saved as {EMB_PREFIX}_{split}.npz
FUSION_CKPT      = "fusion_best_model.pt"            # best image + tabular model
TABULAR_CKPT     = "tabular_best_model.pt"           # best tabular-only model
WRONG_CSV        = "test_wrong_predictions.csv"      # misclassified test cases (fusion model)

# ── Image source: GCS_BASE + 'files/...' part of the cohort local_path ──
#   e.g. gs://.../Datasets/MIMIC-CXR/files/p10/p10000032/s50414267/xxxx.dcm
GCS_BASE = "gs://zarathu-edu-mimic-data-ethereal-mind-460209-e0/Datasets/MIMIC-CXR"

# ── Labels ──
TARGET_COL  = "outcome"                                  # 0 = no transfer, 1 = next-day ICU transfer
CLASS_NAMES = ["no transfer", "next-day ICU transfer"]   # index = label

# ── Tabular features (18 lab tests from 02_build_icu_cohort.R) ──
LAB_COLS = ['hgb', 'hct', 'wbc', 'plt', 'rbc', 'mchc', 'mch', 'mcv', 'rdw',
            'sodium', 'potassium', 'chloride', 'bicarbonate', 'anion_gap',
            'bun', 'creatinine', 'neut_pct', 'lymph_pct']
LAG_COLS = [c + '_lag' for c in LAB_COLS]   # lab features used as model input
REQ_COLS = LAB_COLS + LAG_COLS              # must be non-missing (complete-case)

# ── Sampling (Step 6) ──
N_POS           = 1400    # next-day ICU transfer patients (of 1,428 available)
N_NEG           = 1400    # no-transfer patients → 1:1 balance
VALIDATE_ON_GCS = False   # True: also check file size / DICOM signature (slower)
IO_WORKERS      = 16      # parallel GCS header reads
CHUNK           = 256     # candidates checked per round before early stop
CACHE_SIZE      = 2000    # max images kept by _load_cached (Step 8)

# ── View filter: frontal images only (PA / AP); lateral views excluded ──
ALLOWED_VIEWS  = {"PA", "AP"}
FILTER_BY_VIEW = True

# ── DICOM integrity ──
MIN_DICOM_SIZE = 50 * 1024   # < 50 KB → failed download / corrupt file

# ── Image encoder ──
RAD_DINO    = "microsoft/rad-dino"   # frozen CXR foundation model (Hugging Face)
IMG_EMB_DIM = 768                    # RAD-DINO pooler_output size
IMG_SIZE    = 224                    # display size for the visual check only

# ── Split and training ──
HOLDOUT_FRAC    = 0.30   # val + test share, split equally → 70 / 15 / 15
BATCH_SIZE      = 32
EPOCHS          = 100    # maximum epochs
PATIENCE        = 10     # early stopping on validation ROC-AUC
LR              = 1e-4
WEIGHT_DECAY    = 1e-3
LABEL_SMOOTHING = 0.05
PROJ_DIM        = 64     # per-branch projection size in FusionHead
DROPOUT         = 0.3
NUM_WORKERS     = 0      # DataLoader workers (0 recommended on Windows)
SEED            = 43

# GCS file system (uses the credentials of the authenticated environment)
fs = gcsfs.GCSFileSystem()


def to_gcs_path(local_path: str) -> str:
    """Convert a cohort local_path (.../files/.../xxxx.dcm) to the full GCS path."""
    p = str(local_path).replace("\\", "/")
    idx = p.find("/files/")
    rel = p[idx + 1:] if idx != -1 else p.lstrip("/")   # 'files/...' part
    return f"{GCS_BASE.rstrip('/')}/{rel}"


def set_seed(seed: int):
    """Fix Python / NumPy / PyTorch random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        # deterministic cuDNN so GPU results are reproducible
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

### Step 2. Cohort preprocessing (complete-case)

Reads the cohort exported by `02_build_icu_cohort.R` (`COHORT_CSV`, one index CXR per patient) and prepares it for modelling.

1. **Sex encoding**: `MALE` → 0, `FEMALE` → 1. Value counts are printed before and after mapping; any unexpected label shows up as `NaN` after mapping.
2. **Complete-case filter**: patients missing any column in `REQ_COLS` are removed. The number and share of removed patients and the remaining class counts are printed.
3. The result is saved as `PREPROCESSED_CSV`, the input of Step 3.

In [ ]:
df = pd.read_csv(COHORT_CSV)

# Sex: MALE = 0, FEMALE = 1 (check for NaN after mapping)
print(df['sex'].value_counts(dropna=False))
df['sex'] = df['sex'].str.strip().str.upper().map({'MALE': 0, 'FEMALE': 1})
print(df['sex'].value_counts(dropna=False))

# Complete-case filter
n0 = len(df)
df = df.dropna(subset=REQ_COLS).reset_index(drop=True)
print(f"{n0} -> {len(df)}  (removed {n0 - len(df)}, {1 - len(df)/n0:.1%})")
print(df[TARGET_COL].value_counts())

df.to_csv(PREPROCESSED_CSV, index=False, encoding='utf-8-sig')

### Step 3. Build the sample table

`load_dataframe()` reads `PREPROCESSED_CSV` and prepares one row per patient (one CXR per row):

- `local_path` is converted to the full GCS path with `to_gcs_path()`.
- `file_key` (the DICOM file name) is added for the error-analysis table in Step 14.
- `person_id` and `image_occurrence_id` are cast to integers.

The label stays in `outcome` (0 = no transfer, 1 = next-day ICU transfer). Class counts, the number of unique patients and the positive rate are printed, with a warning if one class is empty.

In [ ]:
def load_dataframe(csv_path: str = PREPROCESSED_CSV) -> pd.DataFrame:
    """Read the preprocessed cohort and return the sample table (GCS paths, file_key, integer ids)."""
    d = pd.read_csv(csv_path)
    d["local_path"] = d["local_path"].apply(to_gcs_path)   # cohort path → GCS path

    pos = d[d[TARGET_COL] == 1]
    neg = d[d[TARGET_COL] == 0]
    print(f"[{CLASS_NAMES[1]}] {len(pos):,}")
    print(f"[{CLASS_NAMES[0]}] {len(neg):,}")

    df = pd.concat([pos, neg], ignore_index=True)

    # DICOM file name, reused in the error-analysis table (Step 14)
    df["file_key"] = df["local_path"].apply(lambda p: os.path.basename(str(p)))
    df["person_id"] = df["person_id"].astype(int)
    df["image_occurrence_id"] = df["image_occurrence_id"].astype(int)

    n_pos, n_neg = len(pos), len(neg)
    print(f"[Samples] total {len(df):,} | {CLASS_NAMES[1]} {n_pos:,} | "
          f"{CLASS_NAMES[0]} {n_neg:,} | unique patients {df['person_id'].nunique():,}")
    if n_pos == 0 or n_neg == 0:
        print("  [Warning] One class has no samples. Check the cohort CSV.")
    return df

df = load_dataframe()
print(f"\nTotal samples: {len(df):,} | positive rate: {df[TARGET_COL].mean():.3f}")

### Step 4. Data check

Checks the data before any image is processed:

- `check_data_distribution()` prints the number of images per class and the positive rate.
- `check_dicom_paths()` checks the first `n_check` GCS paths for (1) existence, (2) a file size of at least `MIN_DICOM_SIZE`, and (3) the DICOM signature `DICM` at byte offset 128. Only 4 header bytes are read per file, so the check is fast. Counts per failure type and up to five failing paths are printed.

If no sampled file passes, `GCS_BASE` or the GCS credentials are wrong.

In [ ]:
def check_data_distribution(df: pd.DataFrame) -> None:
    """Print image counts per class and the positive rate."""
    print("=== Dataset Distribution ===")
    for idx, cls in enumerate(CLASS_NAMES):
        count = int((df[TARGET_COL] == idx).sum())
        print(f"- {idx} = {cls}: {count:,} images")
    print(f"- TOTAL: {len(df):,} images | positive rate {df[TARGET_COL].mean():.3f}")


def check_dicom_paths(df: pd.DataFrame, n_check: int = 20, path_col: str = "local_path") -> None:
    """Check the first n_check paths: exists → size ≥ MIN_DICOM_SIZE → 'DICM' at offset 128."""
    paths = df[path_col].tolist()
    sample = paths[:n_check]
    print(f"\n{len(paths):,} paths | checking the first {len(sample)}")
    print("[Example paths]")
    for p in sample[:3]:
        print(f"  {p}")

    n_missing = n_too_small = n_bad_magic = n_ok = 0
    bad_examples = []
    for p in sample:
        # 1) file exists on GCS
        try:
            exists = fs.exists(p)
        except Exception:
            exists = False
        if not exists:
            n_missing += 1
            if len(bad_examples) < 5:
                bad_examples.append(("missing", p))
            continue
        # 2) file size
        try:
            size = fs.size(p)
        except Exception:
            n_missing += 1
            continue
        if size is None or size < MIN_DICOM_SIZE:
            n_too_small += 1
            if len(bad_examples) < 5:
                bad_examples.append((f"size {size}B", p))
            continue
        # 3) DICOM signature (bytes 128–131)
        try:
            with fs.open(p, "rb") as f:
                f.seek(128)
                magic = f.read(4)
        except Exception:
            magic = b""
        if magic != b"DICM":
            n_bad_magic += 1
            if len(bad_examples) < 5:
                bad_examples.append(("no DICM signature", p))
            continue
        n_ok += 1

    print("\n[Check result (sample)]")
    print(f"  OK             : {n_ok}")
    print(f"  Missing        : {n_missing}")
    print(f"  Too small      : {n_too_small}  (< {MIN_DICOM_SIZE//1024}KB)")
    print(f"  Not DICOM      : {n_bad_magic}")
    if bad_examples:
        print("\n[Failing paths]")
        for reason, p in bad_examples:
            print(f"  ({reason}) {p}")
    if n_ok == 0:
        print("\n[Warning] No valid DICOM in the sample. Check GCS_BASE and credentials.")


check_data_distribution(df)
check_dicom_paths(df, n_check=20)

### Step 5. DICOM reading and view-filter functions

| Function | Role |
|---|---|
| `read_dicom_uint8()` | Reads a DICOM from GCS. MONOCHROME1 images (high value = dark) are inverted to the MONOCHROME2 convention, then pixels are min-max scaled to 0–255 (uint8). Shared by the visual check (Step 7) and the model input (Step 9). |
| `is_valid_dicom()` | Fast integrity check: file size ≥ `MIN_DICOM_SIZE` and `DICM` signature at offset 128 |
| `get_view_position()` | Reads only the DICOM header and returns `ViewPosition` (0018,5101). If it is empty, the view is inferred from `SeriesDescription`, `ProtocolName` or `StudyDescription`, checking `LAT` first so that "LATERAL PA" is not read as PA. |
| `passes_filters()` | Integrity check followed by the view check (cheap check first) |
| `load_dicom_image()` | Display version: `read_dicom_uint8()` → resize to `IMG_SIZE` → grayscale replicated to 3 channels |

No windowing or contrast enhancement is applied, so the visual check shows the same intensities the model receives before RAD-DINO's own resizing and normalization.

In [ ]:
def read_dicom_uint8(path: str) -> np.ndarray:
    """Read a DICOM from GCS → 2-D uint8 image (MONOCHROME1 inverted, min-max scaled to 0–255)."""
    with fs.open(path, "rb") as f:
        ds = pydicom.dcmread(f)
    arr = ds.pixel_array.astype(np.float32)
    # MONOCHROME1: high value = dark → invert to MONOCHROME2
    if getattr(ds, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
        arr = arr.max() - arr
    lo, hi = arr.min(), arr.max()
    arr = np.zeros_like(arr) if hi <= lo else (arr - lo) / (hi - lo) * 255.0
    return arr.astype(np.uint8)


def is_valid_dicom(path: str) -> bool:
    """File size ≥ MIN_DICOM_SIZE and 'DICM' signature. Used via passes_filters() when VALIDATE_ON_GCS=True."""
    try:
        size = fs.size(path)
        if size is None or size < MIN_DICOM_SIZE:
            return False
        with fs.open(path, "rb") as f:
            f.seek(128)                    # DICOM signature starts at offset 128
            return f.read(4) == b"DICM"
    except Exception:
        return False


def get_view_position(path: str) -> Optional[str]:
    """Upper-case ViewPosition from the DICOM header (pixels not read); fallback to description tags."""
    try:
        with fs.open(path, "rb") as f:
            dcm = pydicom.dcmread(f, stop_before_pixels=True, force=True)
    except Exception:
        return None
    vp = getattr(dcm, "ViewPosition", None)
    if vp:
        return str(vp).strip().upper()
    # ViewPosition empty → infer from description tags
    for attr in ("SeriesDescription", "ProtocolName", "StudyDescription"):
        val = getattr(dcm, attr, None)
        if val:
            u = str(val).upper()
            if "LAT" in u: return "LATERAL"   # checked first: 'LATERAL PA' is lateral
            if "PA"  in u: return "PA"
            if "AP"  in u: return "AP"
    return None


def passes_filters(path: str) -> bool:
    """Valid DICOM and (if FILTER_BY_VIEW) an allowed view. Cheap check first."""
    if not is_valid_dicom(path):
        return False
    if not FILTER_BY_VIEW:
        return True
    return get_view_position(path) in ALLOWED_VIEWS


def load_dicom_image(dicom_path: str, img_size: int = IMG_SIZE) -> np.ndarray:
    """DICOM → (img_size, img_size, 3) uint8 image for display (grayscale copied to 3 channels)."""
    data = cv2.resize(read_dicom_uint8(dicom_path), (img_size, img_size))
    return np.stack([data] * 3, axis=-1)

### Step 6. View filtering and class-balanced sampling

`sample_by_view()` selects `N_POS` next-day ICU transfer patients and `N_NEG` no-transfer patients whose index CXR is a frontal view (PA or AP):

1. Each class is shuffled with `SEED`.
2. Candidates are checked in rounds of `CHUNK`, reading DICOM headers in parallel with `IO_WORKERS` threads.
3. Checking stops as soon as the target count is reached, so most headers are never read.
4. If fewer patients pass than requested, a warning is printed and all passing patients are kept.
5. Both classes are combined and shuffled again.

With `N_POS = N_NEG = 1400` the sample is 1:1, so every split in Step 9 has a positive rate of about 0.5. With `VALIDATE_ON_GCS = True`, `passes_filters()` also checks file integrity (slower).

In [ ]:
def _passes(path: str) -> bool:
    """View filter, plus the GCS integrity check when VALIDATE_ON_GCS=True."""
    if VALIDATE_ON_GCS:
        return passes_filters(path)
    if not FILTER_BY_VIEW:
        return True
    return get_view_position(path) in ALLOWED_VIEWS


def sample_by_view(df: pd.DataFrame, n_pos: int, n_neg: int) -> pd.DataFrame:
    """Return n_pos positives and n_neg negatives that pass the view filter, shuffled."""

    def pick(cls: int, target: int) -> pd.DataFrame:
        pool = (df[df[TARGET_COL] == cls]
                .sample(frac=1, random_state=SEED)
                .reset_index(drop=True))

        # Check candidates chunk by chunk in parallel; stop once the target is reached
        kept = []
        rows = pool.to_dict("records")
        with ThreadPoolExecutor(max_workers=IO_WORKERS) as ex:
            for start in range(0, len(rows), CHUNK):
                batch = rows[start:start + CHUNK]
                oks = list(ex.map(lambda r: _passes(r["local_path"]), batch))
                kept.extend(r for r, ok in zip(batch, oks) if ok)
                if len(kept) >= target:
                    break

        got = pd.DataFrame(kept[:target])
        if len(got) < target:
            print(f"[Warning] class {cls} ({CLASS_NAMES[cls]}): {len(got)} passed / {target} requested")
        return got

    print(f"View filtering and sampling... "
          f"(GCS check: {VALIDATE_ON_GCS}, view filter: {FILTER_BY_VIEW})")
    pos = pick(1, n_pos)
    neg = pick(0, n_neg)

    # Shuffle again: rows are grouped by class after concat
    return (pd.concat([pos, neg])
              .sample(frac=1, random_state=SEED)
              .reset_index(drop=True))


available_df = sample_by_view(df, N_POS, N_NEG)
print(f"Final: {CLASS_NAMES[1]} {(available_df[TARGET_COL]==1).sum()} | "
      f"{CLASS_NAMES[0]} {(available_df[TARGET_COL]==0).sum()} | "
      f"total {len(available_df)} | positive rate {available_df[TARGET_COL].mean():.3f}")

### Step 7. Visual check of samples

`visualize_samples()` shows the first `n` images of one class (0 = no transfer, 1 = next-day ICU transfer) after `load_dicom_image()`. Use it to catch black, blank or inverted images before embeddings are extracted.

In [ ]:
%matplotlib inline
def visualize_samples(df: pd.DataFrame, cls_idx: int = 1, n: int = 4) -> None:
    """Show the first n images of one class (0 = no transfer, 1 = next-day ICU transfer)."""
    cls_name = CLASS_NAMES[cls_idx]
    subset = df[df[TARGET_COL] == cls_idx]["local_path"].tolist()[:n]

    ncol = min(4, len(subset))
    nrow = (len(subset) + ncol - 1) // ncol     # grid sized to the number of images
    plt.figure(figsize=(15, 4 * nrow))
    for i, path in enumerate(subset):
        img = load_dicom_image(path)      # (H, W, 3) uint8
        plt.subplot(nrow, ncol, i + 1)
        plt.imshow(img)
        plt.title(f"{cls_name} sample {i+1}")
        plt.axis("off")
    plt.suptitle(f"Visualization: {cls_name}", fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_samples(available_df, cls_idx=0, n=4)   # 0 = no transfer
visualize_samples(available_df, cls_idx=1, n=4)   # 1 = next-day ICU transfer

### Step 8. Optional in-memory image cache

`_load_cached()` wraps `load_dicom_image()` in an LRU cache of up to `CACHE_SIZE` images, so repeated visual inspection of the same files does not re-read them from GCS.

The training pipeline does not use this cache: Step 10 reads each image once to compute its RAD-DINO embedding and saves the embeddings to disk.

In [ ]:
@lru_cache(maxsize=CACHE_SIZE)
def _load_cached(path):
    """load_dicom_image() with an in-memory LRU cache."""
    return load_dicom_image(path)

### Step 9. Dataset, data split and loaders

**`CXRDataset`** converts one row into model input: `read_dicom_uint8()` → RGB → RAD-DINO image processor (resize and center-crop to 518 × 518, then normalization). It returns `(pixel tensor, person_id)`. There is no augmentation, because the encoder is frozen and each image is embedded only once.

**Split**: `available_df` is split into train / validation / test = 70 / 15 / 15, stratified on `outcome`, with `HOLDOUT_FRAC` and `SEED`. The cohort has exactly one row per patient, so this row-level split is also a patient-level split; the assertions confirm that no `person_id` appears in two splits.

**Loaders** use `shuffle=False`, so the order of the embeddings matches the order of the split tables.

In [ ]:
class CXRDataset(Dataset):
    """Rows → (RAD-DINO pixel tensor, person_id); no augmentation."""
    def __init__(self, paths, ids, processor):
        self.paths, self.ids, self.proc = paths, ids, processor

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.fromarray(read_dicom_uint8(self.paths[i])).convert("RGB")
        px = self.proc(images=img, return_tensors="pt").pixel_values[0]  # [3, 518, 518]
        return px, self.ids[i]

In [ ]:
# Stratified 70 / 15 / 15 split; one row per patient → patient-level split
tr, tmp = train_test_split(available_df, test_size=HOLDOUT_FRAC,
                           stratify=available_df[TARGET_COL], random_state=SEED)
va, te  = train_test_split(tmp, test_size=0.50, stratify=tmp[TARGET_COL], random_state=SEED)

for name, d in [("train", tr), ("val", va), ("test", te)]:
    print(name, len(d), d[TARGET_COL].mean().round(4))

# No patient shared between splits
assert len(set(tr.person_id) & set(va.person_id)) == 0
assert len(set(tr.person_id) & set(te.person_id)) == 0
assert len(set(va.person_id) & set(te.person_id)) == 0

In [ ]:
processor = AutoImageProcessor.from_pretrained(RAD_DINO)

def make_ds(d):
    return CXRDataset(d["local_path"].tolist(),
                      d["person_id"].astype(str).tolist(),
                      processor)

ds_tr, ds_va, ds_te = make_ds(tr), make_ds(va), make_ds(te)

# shuffle=False keeps embedding order = table order
common = dict(batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

dl_tr = DataLoader(ds_tr, **common)
dl_va = DataLoader(ds_va, **common)
dl_te = DataLoader(ds_te, **common)

### Step 10. Image embedding extraction (frozen RAD-DINO)

RAD-DINO is used as a fixed feature extractor and is never trained.

- `extract()` runs the encoder in inference mode and keeps `pooler_output`, the CLS token after the final layer norm (`IMG_EMB_DIM = 768`). On GPU it uses fp16 autocast; on CPU it runs in fp32.
- `cache_embeddings()` checks that the embedding order matches the split table (by `person_id`) and saves `{EMB_PREFIX}_{split}.npz` with `emb` (embeddings), `pid` (person_id) and `y` (outcome).
- The three split tables are saved as `SPLIT_CSV` for reproducibility and later analysis.

Later steps read only the `.npz` files, so this step does not need to be repeated once they exist, as long as Steps 1–9 give the same split (same data and `SEED`; Step 11 asserts the order).

In [ ]:
@torch.inference_mode()
def extract(dl, encoder):
    """Return RAD-DINO embeddings (N, IMG_EMB_DIM) and the person_id order."""
    embs, keys = [], []
    for px, pid in tqdm(dl):
        px = px.to(device, non_blocking=True)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=(device == "cuda")):
            out = encoder(pixel_values=px)
        embs.append(out.pooler_output.float().cpu())   # CLS token after final layer norm
        keys += list(pid)
    return torch.cat(embs).numpy(), keys

def cache_embeddings(splits, encoder, prefix=EMB_PREFIX, outdir="."):
    """splits: {"train": (df, dataloader), ...} → saves {prefix}_{split}.npz; returns the file paths."""
    os.makedirs(outdir, exist_ok=True)
    paths = {}
    for name, (d, dl) in tqdm(splits.items()):
        emb, keys = extract(dl, encoder)
        assert keys == d["person_id"].astype(str).tolist(), f"{name}: order mismatch"
        p = os.path.join(outdir, f"{prefix}_{name}.npz")
        np.savez(p, emb=emb, pid=np.array(keys), y=d[TARGET_COL].values)
        paths[name] = p
        print(f"{name}: {emb.shape} -> {p}")
    return paths

In [ ]:
encoder = AutoModel.from_pretrained(RAD_DINO).eval().to(device)   # frozen, inference only

splits = {"train": (tr, dl_tr), "val": (va, dl_va), "test": (te, dl_te)}
cache_embeddings(splits, encoder)

In [ ]:
# Save the split tables (reproducibility, later analysis)
for name, d in [("train", tr), ("val", va), ("test", te)]:
    d.to_csv(SPLIT_CSV.format(name), index=False, encoding='utf-8-sig')

### Step 11. Tabular features and training loaders

**Tabular features** (`build_tab()`):

- Numeric features: the 18 lab features (`LAG_COLS`) and age
- Missing values are filled with the **training-set medians** (a safeguard; the complete-case filter in Step 2 already removes missing lab features).
- Numeric features are standardized with a `StandardScaler` **fitted on the training set only**. Validation and test reuse the training medians and scaler, so no information leaks from them.
- Sex (0/1) is appended unscaled, giving `TAB_DIM` = 18 + 1 + 1 = 20.

**Loaders** (`make_loader()`): each split's cached image embeddings, tabular features and `outcome` are combined into a `TensorDataset`. An assertion checks that the `.npz` patient order matches the split table. The training loader is shuffled; validation and test are not.

`dl_tr`, `dl_va` and `dl_te` are replaced here: from this point they yield `(embedding, tabular, label)` instead of images.

In [ ]:
def build_tab(d, scaler=None, medians=None):
    """Tabular matrix: lab features + age (median-imputed, standardized) + sex (0/1)."""
    X = d[LAG_COLS + ["age"]].copy()
    if medians is None:
        medians = X.median()
    X = X.fillna(medians).values.astype(np.float32)
    if scaler is None:
        scaler = StandardScaler().fit(X)
    X = scaler.transform(X).astype(np.float32)
    sex = d["sex"].astype(np.float32).values[:, None]
    return np.hstack([X, sex]), scaler, medians

# Medians and scaler fitted on train only
Xtr, scaler, medians = build_tab(tr)
Xva, _, _ = build_tab(va, scaler, medians)
Xte, _, _ = build_tab(te, scaler, medians)
TAB_DIM = Xtr.shape[1]

In [ ]:
def make_loader(split, X, d, shuffle):
    """(cached embedding, tabular features, label) loader for one split."""
    z = np.load(f"{EMB_PREFIX}_{split}.npz")
    assert list(z["pid"]) == d["person_id"].astype(str).tolist()   # same patient order
    ds = TensorDataset(torch.from_numpy(z["emb"]),
                       torch.from_numpy(X),
                       torch.from_numpy(d[TARGET_COL].values.astype(np.float32)))
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS)

dl_tr = make_loader("train", Xtr, tr, shuffle=True)
dl_va = make_loader("val",   Xva, va, shuffle=False)
dl_te = make_loader("test",  Xte, te, shuffle=False)

### Step 12. Fusion model and training

**`FusionHead`**

- Image branch: embedding (768) → Linear → GELU → LayerNorm → `PROJ_DIM` (64)
- Tabular branch: tabular features (`TAB_DIM`) → Linear → GELU → LayerNorm → `PROJ_DIM` (64)
- The two branches are concatenated → Dropout (`DROPOUT`) → Linear → 2 logits (0 = no transfer, 1 = next-day ICU transfer)
- With `use_image=False` only the tabular branch is built and the image embedding is ignored. This is the tabular-only baseline used to measure what the image adds.

**Training** (`train_model()`)

- Cross-entropy loss with label smoothing (`LABEL_SMOOTHING`) and the AdamW optimizer (`LR`, `WEIGHT_DECAY`)
- Up to `EPOCHS` epochs, with early stopping when validation ROC-AUC has not improved for `PATIENCE` epochs
- The best model by validation ROC-AUC is saved to `FUSION_CKPT` or `TABULAR_CKPT`.
- `set_seed(SEED)` at the start of each run gives both models the same initialization and shuffling, so each run is reproducible on its own.

`run_epoch()` trains when an optimizer is passed and only evaluates otherwise. It returns the mean loss, accuracy, ROC-AUC and average precision (AP), and the training log prints them per epoch.

In [ ]:
class FusionHead(nn.Module):
    """Image embedding + tabular features → 2-class logits; use_image=False → tabular-only."""
    def __init__(self, img_dim=IMG_EMB_DIM, tab_dim=TAB_DIM, proj_dim=PROJ_DIM,
                 dropout=DROPOUT, use_image=True):
        super().__init__()
        self.use_image = use_image
        if use_image:
            self.reduce = nn.Sequential(
                nn.Linear(img_dim, proj_dim), nn.GELU(), nn.LayerNorm(proj_dim))
        self.expand = nn.Sequential(
            nn.Linear(tab_dim, proj_dim), nn.GELU(), nn.LayerNorm(proj_dim))
        d = proj_dim * 2 if use_image else proj_dim
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d, len(CLASS_NAMES)))

    def forward(self, img_emb, tab):
        z = self.expand(tab)
        if self.use_image:
            z = torch.cat([self.reduce(img_emb), z], dim=1)
        return self.head(z)

In [ ]:
def run_epoch(model, dl, crit, opt=None):
    """One pass over dl; trains if opt is given. Returns (mean loss, accuracy, ROC-AUC, AP)."""
    train = opt is not None
    model.train(train)

    tot, correct, ys, ps = 0.0, 0, [], []

    with torch.set_grad_enabled(train):
        for emb, tab, y in dl:
            emb, tab, y = emb.to(device), tab.to(device), y.to(device).long()

            logit = model(emb, tab)
            loss = crit(logit, y)

            if train:
                opt.zero_grad()
                loss.backward()
                opt.step()

            tot += loss.item() * len(y)
            correct += (logit.argmax(1) == y).sum().item()

            ys.append(y.cpu())
            ps.append(torch.softmax(logit, 1)[:, 1].detach().cpu())   # P(next-day ICU transfer)

    y = torch.cat(ys).numpy()
    p = torch.cat(ps).numpy()

    return tot / len(y), correct / len(y), roc_auc_score(y, p), average_precision_score(y, p)

In [ ]:
def train_model(use_image: bool, epochs: int = EPOCHS, patience: int = PATIENCE) -> float:
    """Train FusionHead with early stopping on validation ROC-AUC; save the best checkpoint; return best AUC."""
    set_seed(SEED)   # reproducible initialization and shuffling per run

    model = FusionHead(tab_dim=TAB_DIM, use_image=use_image).to(device)
    crit = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    ckpt = FUSION_CKPT if use_image else TABULAR_CKPT

    best, bad = -1, 0

    for ep in range(epochs):
        tr_loss, _, _, _ = run_epoch(model, dl_tr, crit, opt)
        _, va_acc, va_auc, va_ap = run_epoch(model, dl_va, crit)
        print(f"epoch {ep:3d} | train loss {tr_loss:.4f} | "
              f"val acc {va_acc:.4f} | val AUC {va_auc:.4f} | val AP {va_ap:.4f}")

        if va_auc > best:
            best, bad = va_auc, 0
            torch.save(model.state_dict(), ckpt)   # best model so far
        else:
            bad += 1
            if bad >= patience:
                break

    print(f"best val AUC {best:.4f} -> {ckpt}")
    return best

In [ ]:
train_model(use_image=True)    # image + tabular (fusion)

In [ ]:
train_model(use_image=False)   # tabular-only baseline

### Step 13. Test-set evaluation

`evaluate_checkpoint()` loads a saved best model, predicts the test set once and prints:

- the confusion matrix (rows = true label, columns = predicted label)
- the classification report (precision, recall and F1 per class)
- ROC-AUC, F1 for the positive class (next-day ICU transfer) and accuracy

The predicted label is the class with the larger logit, which is equivalent to a probability threshold of 0.5. The fusion results are kept as `y_true_f`, `y_pred_f`, `y_prob_f` and the tabular-only results as `y_true_t`, `y_pred_t`, `y_prob_t`. Both ROC curves are drawn on the same plot to compare the two models.

In [ ]:
@torch.no_grad()
def evaluate_model(loader, model):
    """Return (true labels, predicted labels, P(next-day ICU transfer)) over the whole loader."""
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for emb, tab, y in loader:
        emb, tab, y = emb.to(device), tab.to(device), y.to(device).long()
        logit = model(emb, tab)
        probs = torch.softmax(logit, dim=1)[:, 1]
        preds = logit.argmax(dim=1)
        all_labels.extend(y.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)


def evaluate_checkpoint(ckpt_path: str, use_image: bool):
    """Load a best checkpoint, evaluate on the test set, print metrics; return (y_true, y_pred, y_prob)."""
    model = FusionHead(tab_dim=TAB_DIM, use_image=use_image).to(device)
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    y_true, y_pred, y_prob = evaluate_model(dl_te, model)

    print("[Confusion Matrix]  (rows = true, columns = predicted)")
    print(confusion_matrix(y_true, y_pred))
    print("\n[Classification Report]")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))

    # ROC-AUC is undefined if only one class is present
    try:
        auroc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auroc = float("nan")
    print(f"ROC-AUC : {auroc:.4f}")
    print(f"F1      : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    return y_true, y_pred, y_prob


# Fusion model (image + tabular)
y_true_f, y_pred_f, y_prob_f = evaluate_checkpoint(FUSION_CKPT, use_image=True)

In [ ]:
# Tabular-only baseline
y_true_t, y_pred_t, y_prob_t = evaluate_checkpoint(TABULAR_CKPT, use_image=False)

# ROC curves of both models on the test set
plt.figure(figsize=(5, 5))
for name, yt, yp in [("Fusion", y_true_f, y_prob_f), ("Tabular-only", y_true_t, y_prob_t)]:
    fpr, tpr, _ = roc_curve(yt, yp)
    plt.plot(fpr, tpr, label=f"{name} AUROC={roc_auc_score(yt, yp):.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")   # chance line
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC (test set)"); plt.legend()
plt.show()

### Step 14. Error analysis (fusion model)

Lists the test-set cases the fusion model got wrong, **most confident errors first**, ranked by |probability − 0.5|:

- **FP (false positive)**: predicted next-day ICU transfer, but no transfer occurred. This leads to unnecessary escalation or extra checks.
- **FN (false negative)**: predicted no transfer, but the patient was transferred to the ICU the next day. This is the clinically most dangerous error.

Confident errors are the most informative cases to review, such as hard cases or possible label errors. The table is saved as `WRONG_CSV` with `utf-8-sig` encoding so it opens correctly in Excel.

> Row `i` of the predictions corresponds to `te.iloc[i]` because `dl_te` is not shuffled.

In [ ]:
# Fusion model test results (Step 13)
y_true, y_pred, y_prob = y_true_f, y_pred_f, y_prob_f

# Misclassified cases, most confident first (larger |prob − 0.5|)
wrong_idx = np.where(y_pred != y_true)[0]
wrong_idx = sorted(wrong_idx, key=lambda i: abs(y_prob[i] - 0.5), reverse=True)

fp = int(((y_pred == 1) & (y_true == 0)).sum())   # false positive
fn = int(((y_pred == 0) & (y_true == 1)).sum())   # false negative
print(f"Misclassified: {len(wrong_idx)} / {len(y_true)}")
print(f"  False positive (predicted transfer, no transfer)        : {fp}")
print(f"  False negative (missed next-day ICU transfer)           : {fn}")

wrong_df = pd.DataFrame({
    "person_id":           te.iloc[wrong_idx]["person_id"].values,
    "image_occurrence_id": te.iloc[wrong_idx]["image_occurrence_id"].values,
    "file_name":           te.iloc[wrong_idx]["file_key"].values,
    "true":                y_true[wrong_idx],
    "prob":                np.round(y_prob[wrong_idx], 4),
    "pred":                y_pred[wrong_idx],
    "error_type":          np.where(y_pred[wrong_idx] == 1,
                                    "FP (predicted transfer, no transfer)",
                                    "FN (missed next-day ICU transfer)"),
})

# utf-8-sig: opens correctly in Excel
wrong_df.to_csv(WRONG_CSV, index=False, encoding="utf-8-sig")
print(f"\nSaved: {WRONG_CSV} ({len(wrong_df)} rows)")
wrong_df.head(10)